# Criteo 1TB Click Logs — Exploratory Data Analysis

Goal: understand the data well enough to justify the modelling choices (features, encoding, metrics, validation).

**Scope:** only the training days (2015-02-15 and 2015-02-16) are explored here. Validation and test days are kept untouched to avoid leaking information into modelling decisions — the only exception is the drift check in section 7, which compares the two training days with each other.

Data: one parquet part-file per day, downloaded with `scripts/download_data.py`.

## 0. Setup

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

In [2]:
DATA_DIR = Path("../data/raw/data")
TRAIN_DAYS = ["2015-02-15", "2015-02-16"]

LABEL = "label"
INT_COLS = [f"integer_feature_{i}" for i in range(1, 14)]
CAT_COLS = [f"categorical_feature_{i}" for i in range(1, 27)]


def load_days(days: list[str]) -> pd.DataFrame:
    """Load every parquet file of the given days and tag rows with their day."""
    frames = []
    for day in days:
        for path in sorted((DATA_DIR / f"day={day}").glob("*.parquet")):
            frames.append(pd.read_parquet(path).assign(day=day))
    return pd.concat(frames, ignore_index=True)


df = load_days(TRAIN_DAYS)
print(f"{len(df):,} rows x {df.shape[1]} columns")
df.head()

2,308,265 rows x 41 columns


,label,integer_feature_1,integer_feature_2,integer_feature_3,integer_feature_4,integer_feature_5,integer_feature_6,integer_feature_7,integer_feature_8,integer_feature_9,integer_feature_10,integer_feature_11,integer_feature_12,integer_feature_13,categorical_feature_1,categorical_feature_2,categorical_feature_3,categorical_feature_4,categorical_feature_5,categorical_feature_6,categorical_feature_7,categorical_feature_8,categorical_feature_9,categorical_feature_10,categorical_feature_11,categorical_feature_12,categorical_feature_13,categorical_feature_14,categorical_feature_15,categorical_feature_16,categorical_feature_17,categorical_feature_18,categorical_feature_19,categorical_feature_20,categorical_feature_21,categorical_feature_22,categorical_feature_23,categorical_feature_24,categorical_feature_25,categorical_feature_26,day
0,0,NaN,84.0000,NaN,31.0000,12.0000,0.0000,0.0000,1,8,0.0000,3.0000,28899.0000,NaN,8a2b1e43,68fe2a00,f9b6c7f8,9d24e8e8,c6fc10d3,6fcd6dcb,41c974b7,9f591967,2e4e821f,31184e3d,fa478aa9,96aaa628,a77a4a56,NaN,39100785,NaN,NaN,f7389918,139221a3,8024f45f,0683bc6f,fa7eca6c,NaN,e716200f,30436bfc,2ccea557,2015-02-15
1,0,104.0000,1045.0000,30.0000,NaN,1.0000,0.0000,0.0000,1,9,0.0000,1.0000,1527.0000,30.0000,ad98e872,3dbb483e,652f15b0,6521d620,c6fc10d3,6fcd6dcb,674d9101,54cd7262,7cee8453,62da11e3,14874876,4a914b5d,a77a4a56,be4ee537,a63cedcf,4cdc3efa,d20856aa,6785af47,cc7a7d35,156cbe87,96fbe197,15562d5d,d3df7183,c91256e2,30436bfc,e1be5ef2,2015-02-15
2,0,6.0000,145.0000,10.0000,NaN,4.0000,0.0000,0.0000,0,12,0.0000,2.0000,17366.0000,10.0000,788a5d5b,98e2c109,921258ba,f2463ffb,c6fc10d3,6fcd6dcb,e856635a,a66a02fa,2e4e821f,16ec16d6,fa478aa9,8d2db913,a77a4a56,NaN,ded4997d,NaN,NaN,a1eb1511,9512c20b,ef426d46,0683bc6f,6e86ac23,NaN,af97a0c1,30436bfc,b757e957,2015-02-15
3,0,40.0000,360.0000,7.0000,NaN,9.0000,0.0000,0.0000,312,0,0.0000,4.0000,18.0000,7.0000,ad98e872,cea68cd3,d0c5d50f,530c3ed4,c6fc10d3,6fcd6dcb,725b10e7,c0375b13,2e4e821f,62da11e3,14874876,06d701ea,a77a4a56,be4ee537,fdc9e405,4cdc3efa,d20856aa,b8170bba,cc7a7d35,156cbe87,96fbe197,15562d5d,d3df7183,245d9da7,30436bfc,e1be5ef2,2015-02-15
4,0,NaN,339.0000,NaN,NaN,1.0000,0.0000,0.0000,248,7,0.0000,1.0000,28740.0000,NaN,9778ef3a,32934749,3a92e2e0,764c8142,c6fc10d3,6fcd6dcb,9d4501a9,edb0ae6c,2e4e821f,22f47d42,bf694354,12716184,00c5ffb7,NaN,60537734,NaN,NaN,7232d217,9512c20b,14cde947,45f35b4b,f52bfc8b,NaN,edc6442f,ff654802,2ccea557,2015-02-15


In [3]:
df.dtypes.value_counts()

str        27
float64    11
int32       3
Name: count, dtype: int64

In [4]:
def memory_gb(frame: pd.DataFrame) -> float:
    return frame.memory_usage(deep=True).sum() / 1e9


mem_before = memory_gb(df)
df[CAT_COLS] = df[CAT_COLS].astype("category")
mem_after = memory_gb(df)
print(f"Memory with string columns:   {mem_before:.2f} GB")
print(f"Memory with category columns: {mem_after:.2f} GB  ({mem_before / mem_after:.1f}x smaller)")

Memory with string columns:   1.21 GB
Memory with category columns: 0.45 GB  (2.7x smaller)


**Takeaways**
- 2.3M training rows, 13 integer + 26 categorical features.
- Integer columns containing nulls are loaded as `float64` (pandas has no nullable int by default); only `integer_feature_8` and `integer_feature_9` have no missing values and stay `int32`.
- The 26 hashed categorical columns dominate memory. Converting them to `category` makes the frame **2.7x smaller** (1.21 GB → 0.45 GB) → do this systematically in the training pipeline.